# Week 2: The Mechanics of Meaning (Tokenization)

This notebook builds intuition for tokenization before implementing tokenizers from scratch.

## Day 1 Goals
1. Compare **word-level**, **character-level**, and **subword** tokenization.
2. Understand why OOV (out-of-vocabulary) happens and how subword methods reduce it.
3. Introduce **byte-level BPE** and why it is robust for emojis, symbols, and multilingual text.
4. Separate **tokenization** from **model input formatting** (special tokens).
5. End with a clear comparison table and practical takeaways for LLMs.

In [22]:
import warnings
warnings.filterwarnings("ignore")

import random
from collections import Counter 

from transformers import AutoTokenizer
from utils import get_imdb_corpus, summarize_one, corpus_token_lengths, word_tokenizer, character_tokenizer, word_piece_tokenizer, bpe_tokenizer, train_bpe, train_wordpiece, byte_level_tokenizer
from dotenv import load_dotenv

load_dotenv("../../.env")

True

## 0. Data Setup: IMDB + Stress Test Set

We use IMDB as the main corpus because it is realistic English text with varied vocabulary.

To test robustness, we also add a small stress set with:
- emojis
- accented words
- CJK text
- URLs/hashtags
- punctuation-heavy strings

This combination helps us see where each tokenization strategy breaks or succeeds.


In [4]:
imdb_corpus = get_imdb_corpus()
print(f"Total IMDB reviews: {len(imdb_corpus)}")

random.seed(42)
sample_text = random.choice(imdb_corpus)
print(sample_text[:600])

Total IMDB reviews: 100000
We love movies of all kinds. This movie was filmed in Finland by a Finnish director who appears to be trying to imitate spy movies from James Bond to Mission Impossible...and they did a very poor job of imitating. <br /><br />Sadly, some really good American actors worked on this film. It was apparently made as a serious movie, but even Netflix lists this as a spy 'farce'.<br /><br />If you like the Scary Movie or Airplane type movies, you might like this one OK - no way you could like it that much.<br /><br />We really like Bill Pullman, but this movie would made you doubt his movie making de


In [3]:
stress_texts = [
    "I loved it 😂🔥",
    "café naïve résumé",
    "今天天气很好",
    "Check https://example.com #NLP",
    "price=$19.99, don't miss it!",
]

## 1. Word Boundary Tokenization

Word-level tokenization splits text into words (usually by whitespace and punctuation rules).

### Why people use it
- Simple and interpretable
- Fast to prototype

### Main limitation
- Large vocabulary
- Poor handling of rare words, misspellings, and unseen forms (OOV)
- Weak robustness for messy text


In [7]:
word_tokens = word_tokenizer(sample_text)
summarize_one("Word Token", sample_text, word_tokens)
print(f"Unique tokens in sample: {len(set(word_tokens))}")


[Word Token]
text: We love movies of all kinds. This movie was filmed in Finland by a Finnish director who ap...
num_tokens: 195
tokens[:20]: ['We', 'love', 'movies', 'of', 'all', 'kinds', '.', 'This', 'movie', 'was', 'filmed', 'in', 'Finland', 'by', 'a', 'Finnish', 'director', 'who', 'appears', 'to']
Unique tokens in sample: 106


In [8]:
for t in stress_texts:
    summarize_one("Word (stress)", t, word_tokenizer(t), max_show=30)


[Word (stress)]
text: I loved it 😂🔥
num_tokens: 5
tokens[:30]: ['I', 'loved', 'it', '😂', '🔥']

[Word (stress)]
text: café naïve résumé
num_tokens: 3
tokens[:30]: ['café', 'naïve', 'résumé']

[Word (stress)]
text: 今天天气很好
num_tokens: 1
tokens[:30]: ['今天天气很好']

[Word (stress)]
text: Check https://example.com #NLP
num_tokens: 10
tokens[:30]: ['Check', 'https', ':', '/', '/', 'example', '.', 'com', '#', 'NLP']

[Word (stress)]
text: price=$19.99, don't miss it!
num_tokens: 13
tokens[:30]: ['price', '=', '$', '19', '.', '99', ',', 'don', "'", 't', 'miss', 'it', '!']


## 2. Character Boundary Tokenization

Character-level tokenization splits text into single characters.

### Strengths
- Very small vocabulary
- Nearly no OOV problem

### Tradeoff
- Sequences become much longer
- Harder for models to capture meaning efficiently
- Higher compute cost for long contexts


In [9]:
char_tokens = character_tokenizer(sample_text)
summarize_one("Character token", sample_text, char_tokens)
print(f"Unique chars in sample: {len(set(char_tokens))}")


[Character token]
text: We love movies of all kinds. This movie was filmed in Finland by a Finnish director who ap...
num_tokens: 764
tokens[:20]: ['W', 'e', ' ', 'l', 'o', 'v', 'e', ' ', 'm', 'o', 'v', 'i', 'e', 's', ' ', 'o', 'f', ' ', 'a', 'l']
Unique chars in sample: 47


In [10]:
for t in stress_texts:
    summarize_one("Character (stress)", t, character_tokenizer(t), max_show=40)


[Character (stress)]
text: I loved it 😂🔥
num_tokens: 13
tokens[:40]: ['I', ' ', 'l', 'o', 'v', 'e', 'd', ' ', 'i', 't', ' ', '😂', '🔥']

[Character (stress)]
text: café naïve résumé
num_tokens: 17
tokens[:40]: ['c', 'a', 'f', 'é', ' ', 'n', 'a', 'ï', 'v', 'e', ' ', 'r', 'é', 's', 'u', 'm', 'é']

[Character (stress)]
text: 今天天气很好
num_tokens: 6
tokens[:40]: ['今', '天', '天', '气', '很', '好']

[Character (stress)]
text: Check https://example.com #NLP
num_tokens: 30
tokens[:40]: ['C', 'h', 'e', 'c', 'k', ' ', 'h', 't', 't', 'p', 's', ':', '/', '/', 'e', 'x', 'a', 'm', 'p', 'l', 'e', '.', 'c', 'o', 'm', ' ', '#', 'N', 'L', 'P']

[Character (stress)]
text: price=$19.99, don't miss it!
num_tokens: 28
tokens[:40]: ['p', 'r', 'i', 'c', 'e', '=', '$', '1', '9', '.', '9', '9', ',', ' ', 'd', 'o', 'n', "'", 't', ' ', 'm', 'i', 's', 's', ' ', 'i', 't', '!']


## 3. Subword Tokenization (WordPiece / BPE Family)

Subword tokenization is a practical middle ground:
- Frequent words can stay whole
- Rare words are split into meaningful pieces

Examples:
- `tokenization` -> `token` + `##ization` (WordPiece-style)
- `unaffable` -> smaller parts instead of full OOV

This is why modern NLP and LLM systems rely on subword methods.

### 3A. WordPiece (BERT-style intuition)

WordPiece builds a vocabulary of subword units and applies greedy longest-match segmentation.

### Practical effect
- Reduces OOV compared to word-level
- Keeps sequence lengths shorter than character-level
- Can still emit `[UNK]` for unsupported symbols/scripts depending on vocab


In [11]:
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

bert_tokens = bert_tokenizer.tokenize(sample_text)
summarize_one("WordPiece/Bert", sample_text, bert_tokens)
print(f"Unique tokens in sample:{len(set(bert_tokens))}")


[WordPiece/Bert]
text: We love movies of all kinds. This movie was filmed in Finland by a Finnish director who ap...
num_tokens: 199
tokens[:20]: ['we', 'love', 'movies', 'of', 'all', 'kinds', '.', 'this', 'movie', 'was', 'filmed', 'in', 'finland', 'by', 'a', 'finnish', 'director', 'who', 'appears', 'to']
Unique tokens in sample:105


In [12]:
for t in stress_texts:
    toks = bert_tokenizer.tokenize(t)
    summarize_one("WordPiece/BERT (stress)", t, toks, max_show=30)
    has_unk = "[UNK]" in toks
    print(f"has_[UNK]={has_unk} | text={t}")


[WordPiece/BERT (stress)]
text: I loved it 😂🔥
num_tokens: 4
tokens[:30]: ['i', 'loved', 'it', '[UNK]']
has_[UNK]=True | text=I loved it 😂🔥

[WordPiece/BERT (stress)]
text: café naïve résumé
num_tokens: 3
tokens[:30]: ['cafe', 'naive', 'resume']
has_[UNK]=False | text=café naïve résumé

[WordPiece/BERT (stress)]
text: 今天天气很好
num_tokens: 6
tokens[:30]: ['[UNK]', '天', '天', '[UNK]', '[UNK]', '[UNK]']
has_[UNK]=True | text=今天天气很好

[WordPiece/BERT (stress)]
text: Check https://example.com #NLP
num_tokens: 11
tokens[:30]: ['check', 'https', ':', '/', '/', 'example', '.', 'com', '#', 'nl', '##p']
has_[UNK]=False | text=Check https://example.com #NLP

[WordPiece/BERT (stress)]
text: price=$19.99, don't miss it!
num_tokens: 13
tokens[:30]: ['price', '=', '$', '19', '.', '99', ',', 'don', "'", 't', 'miss', 'it', '!']
has_[UNK]=False | text=price=$19.99, don't miss it!


### 3B. Byte-Level BPE (GPT-style intuition)

Byte-level BPE starts from UTF-8 bytes, then learns merges.

### Why this matters
- Any text can be represented as bytes
- Extremely robust for emojis, symbols, multilingual text, and noisy inputs
- Avoids hard failures from unknown characters


In [13]:
gpt2_tokenizer = AutoTokenizer.from_pretrained("gpt2")

gpt2_tokens = gpt2_tokenizer.tokenize(sample_text)
summarize_one("Byte-Level BPE/GPT-2", sample_text, gpt2_tokens)
print(f"Unique tokens in sample:{len(set(gpt2_tokens))}")


[Byte-Level BPE/GPT-2]
text: We love movies of all kinds. This movie was filmed in Finland by a Finnish director who ap...
num_tokens: 184
tokens[:20]: ['We', 'Ġlove', 'Ġmovies', 'Ġof', 'Ġall', 'Ġkinds', '.', 'ĠThis', 'Ġmovie', 'Ġwas', 'Ġfilmed', 'Ġin', 'ĠFinland', 'Ġby', 'Ġa', 'ĠFinnish', 'Ġdirector', 'Ġwho', 'Ġappears', 'Ġto']
Unique tokens in sample:118


In [12]:
for t in stress_texts:
    toks = gpt2_tokenizer.tokenize(t)
    summarize_one("Byte-Level BPE/GPT-2 (stress)", t, toks, max_show=30)



[Byte-Level BPE/GPT-2 (stress)]
text: I loved it 😂🔥
num_tokens: 8
tokens[:30]: ['I', 'Ġloved', 'Ġit', 'ĠðŁĺ', 'Ĥ', 'ðŁ', 'Ķ', '¥']

[Byte-Level BPE/GPT-2 (stress)]
text: café naïve résumé
num_tokens: 7
tokens[:30]: ['c', 'af', 'Ã©', 'ĠnaÃ¯ve', 'ĠrÃ©', 'sum', 'Ã©']

[Byte-Level BPE/GPT-2 (stress)]
text: 今天天气很好
num_tokens: 10
tokens[:30]: ['ä»', 'Ĭ', 'å¤©', 'å¤©', 'æ°', 'Ķ', 'å¾', 'Ī', 'å¥', '½']

[Byte-Level BPE/GPT-2 (stress)]
text: Check https://example.com #NLP
num_tokens: 9
tokens[:30]: ['Check', 'Ġhttps', '://', 'example', '.', 'com', 'Ġ#', 'N', 'LP']

[Byte-Level BPE/GPT-2 (stress)]
text: price=$19.99, don't miss it!
num_tokens: 11
tokens[:30]: ['price', '=$', '19', '.', '99', ',', 'Ġdon', "'t", 'Ġmiss', 'Ġit', '!']


## Encoding Primer: ASCII vs Unicode vs UTF-8

- **ASCII**
  A small character set (128 symbols) for basic English letters, digits, punctuation, and control chars.
  Example: `A` = 65.

- **Unicode**
  A universal character standard that assigns each character a unique code point (like `U+0041` for `A`, `U+4E2D` for `中`, `U+1F92A` for `🤪`).

- **UTF-8**
  An encoding format for Unicode code points into bytes (1–4 bytes per character).
  ASCII characters stay 1 byte, so UTF-8 is backward-compatible with ASCII.

**Why tokenization cares:**
Byte-level tokenizers (e.g., GPT-style byte-level BPE) work on UTF-8 bytes, so they can represent any text (emoji, accents, CJK, symbols) without unknown-character failure.


In [14]:
# Quick demo: code points vs UTF-8 bytes
s = "Aé中🤪"

print("Text:", s)
print("Unicode code points:", [f"U+{ord(ch):04X}" for ch in s])
print("UTF-8 bytes:", list(s.encode("utf-8")))


Text: Aé中🤪
Unicode code points: ['U+0041', 'U+00E9', 'U+4E2D', 'U+1F92A']
UTF-8 bytes: [65, 195, 169, 228, 184, 173, 240, 159, 164, 170]


## 4. Special Tokens: Packaging for Model Inputs (Not Raw Corpus)

Important distinction:

- **Tokenization**: split text into token units
- **Sequence packaging**: add model control tokens such as `[CLS]`, `[SEP]`, `<bos>`, `<eos>`, `<pad>`

We add special tokens when preparing model inputs, not when building raw corpus text.

**Input formatting** is the step that turns tokenized text into the exact tensor layout a model expects.

***Purpose:***

- Add structure the model was trained with.
- Make variable-length text batchable.
- Tell the model which tokens are real vs padding.
- Support specific tasks (classification, QA, generation).
- Typical input formatting includes:
    * Add special tokens ([CLS], [SEP], <bos>, <eos>)
    * Convert tokens to IDs
    * Truncate/pad to max_length
    * Build masks:
          * attention_mask (1 real token, 0 padding)
    * sometimes token_type_ids (sentence A/B for BERT-style)
    * Build labels for training objective (e.g., shifted labels for next-token prediction)

In [15]:
text_sample = "Hello world!"

bert_ids_plain = bert_tokenizer.encode(text_sample, add_special_tokens=False)
bert_ids_special = bert_tokenizer.encode(text_sample, add_special_tokens=True)

print("BERT plain tokens:   ", bert_tokenizer.convert_ids_to_tokens(bert_ids_plain))
print("BERT special tokens: ", bert_tokenizer.convert_ids_to_tokens(bert_ids_special))


BERT plain tokens:    ['hello', 'world', '!']
BERT special tokens:  ['[CLS]', 'hello', 'world', '!', '[SEP]']


In [16]:
# GPT-2 often has no default BOS/EOS in simple encode call
gpt2_ids = gpt2_tokenizer.encode(text_sample, add_special_tokens=True)
print("GPT-2 tokens:", gpt2_tokenizer.convert_ids_to_tokens(gpt2_ids))
print(gpt2_tokenizer.special_tokens_map)
print("bos_token:", gpt2_tokenizer.bos_token, "eos_token:", gpt2_tokenizer.eos_token, "pad_token:", gpt2_tokenizer.pad_token)


GPT-2 tokens: ['Hello', 'Ġworld', '!']
{'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>'}
bos_token: <|endoftext|> eos_token: <|endoftext|> pad_token: None


## 5. Quick Comparison on the Same Inputs

For each method, compare:
- Number of tokens (sequence length)
- Whether unknown tokens appear
- Behavior on stress-test texts
- Decoding quality (encode -> decode)

This turns tokenization discussion from theory into measurable behavior.


In [17]:
# Use a small subset for speed
subset = imdb_corpus[:2000]

metrics = {
    "word": corpus_token_lengths(subset, word_tokenizer),
    "char": corpus_token_lengths(subset, character_tokenizer),
    "bert_wordpiece": corpus_token_lengths(subset, lambda t: bert_tokenizer.tokenize(t, truncation=True, max_length=512)),
    "gpt2_byte_bpe": corpus_token_lengths(subset, lambda t: gpt2_tokenizer.tokenize(t, truncation=True, max_length=1024)),
}

print("\n--- Corpus Length Stats (first 2,000 IMDB texts) ---")
for name, m in metrics.items():
    print(f"{name:16} avg={m['avg_len']:<8} median={m['median_len']:<8} min={m['min_len']:<6} max={m['max_len']}")


--- Corpus Length Stats (first 2,000 IMDB texts) ---
word             avg=287.79   median=218.0    min=16     max=1600
char             avg=1276.16  median=964.0    min=65     max=7382
bert_wordpiece   avg=303.84   median=230.0    min=19     max=1681
gpt2_byte_bpe    avg=290.47   median=221.0    min=16     max=1659


In [18]:
print("\n--- Single-text Summary ---")
print(f"Original words:      {len(sample_text.split())}")
print(f"Original characters: {len(sample_text)}")
print(f"Word tokens:         {len(word_tokens)}")
print(f"Character tokens:    {len(char_tokens)}")
print(f"BERT tokens:         {len(bert_tokens)}")
print(f"GPT-2 tokens:        {len(gpt2_tokens)}")


--- Single-text Summary ---
Original words:      138
Original characters: 764
Word tokens:         195
Character tokens:    764
BERT tokens:         199
GPT-2 tokens:        184


In [19]:
comparison_rows = [
    ("Word", "High", "Largest", "Shortest", "Weak", "Weak"),
    ("Character", "Very Low", "Smallest", "Longest", "Strong", "Strong"),
    ("WordPiece", "Lower than word", "Medium", "Medium", "Medium", "Medium"),
    ("Byte-level BPE", "Very Low", "Medium", "Medium", "Strong", "Strong"),
]

for r in comparison_rows:
    print(f"{r[0]:15} | OOV risk={r[1]:16} | Vocab={r[2]:8} | SeqLen={r[3]:8} | Multilingual={r[4]:6} | Emoji/Symbols={r[5]}")


Word            | OOV risk=High             | Vocab=Largest  | SeqLen=Shortest | Multilingual=Weak   | Emoji/Symbols=Weak
Character       | OOV risk=Very Low         | Vocab=Smallest | SeqLen=Longest  | Multilingual=Strong | Emoji/Symbols=Strong
WordPiece       | OOV risk=Lower than word  | Vocab=Medium   | SeqLen=Medium   | Multilingual=Medium | Emoji/Symbols=Medium
Byte-level BPE  | OOV risk=Very Low         | Vocab=Medium   | SeqLen=Medium   | Multilingual=Strong | Emoji/Symbols=Strong


## 6. Summary: What We Learned Today

- Word-level is simple but brittle (OOV-heavy).
- Character-level is robust but inefficient (long sequences).
- Subword methods are the practical compromise.
- Byte-level BPE is especially robust for real-world messy text.
- Special tokens belong to model input formatting, not corpus construction.

Byte-level subword tokenization (e.g., byte-level BPE) is a strong default because it:

has very low OOV risk (any text can be represented via bytes),
keeps a practical balance between vocabulary size and sequence length,
is robust across multilingual text, emojis, and symbols.
That robustness and practicality are key reasons many modern LLM tokenizers use BPE-family methods with byte-level handling.

This prepares us for Day 2: implementing **BPE** and **WordPiece** from scratch.


# Day 2: From-Scratch Tokenizers (WordPiece + BPE)

We will train small WordPiece and BPE tokenizers on a subset of IMDB, then tokenize the same sample text.


## 7. Train a BPE Model (From Scratch)
We build merges from imdb corpus.


### BPE Algorithm
1. pretokenize text into words using word tokenizer.
    Example: "hello world!" -> ["hello", "world", !"]
2. split each word into characters and append end of word marker </w>.
    Example: "hello" -> h e l l o </w>
3. Count Adjacent symbol pairs across the whole corpus, weighted by word freq:
    Example: (h,e), (e,l), (l,l), (l,o), (o, </w>)
4. Find the most frequent pair.
5. Merge that pair into a new symbol everywhere:
   example:  l+l -> ll
6. Repeate step 3 to 5 until:
    1. initial_vocab_size + number of merges >= target vocab size.
    2. no pair frequency exceeds min_freq
    3. you hit a max merge limit
7. result is an ordered list of merges.

Tokenization step:
1. pretokenize word.
2. split each word inot characters with </w>
3. apply learned merges in order to that word
4. remove </w> and return the subword token.

In [23]:
bpe_merges = train_bpe(imdb_corpus[:5000], vocab_size=1000, min_freq=2, byte_level=False)
bpe_tokens_scratch = bpe_tokenizer(sample_text, bpe_merges, byte_level=False)
summarize_one("BPE (scratch)", sample_text, bpe_tokens_scratch)


[BPE (scratch)]
text: We love movies of all kinds. This movie was filmed in Finland by a Finnish director who ap...
num_tokens: 281
tokens[:20]: ['W', 'e</w>', 'love</w>', 'movies</w>', 'of</w>', 'all</w>', 'k', 'in', 'ds</w>', '.</w>', 'This</w>', 'movie</w>', 'was</w>', 'fil', 'm', 'ed</w>', 'in</w>', 'F', 'in', 'l']


### Byte-Level BPE (Scratch)
We can train BPE over bytes instead of characters to guarantee coverage for any input.


In [24]:
bpe_merges_byte = train_bpe(imdb_corpus[:5000], vocab_size=1000, min_freq=2, byte_level=True)
bpe_tokens_byte = bpe_tokenizer(sample_text, bpe_merges_byte, byte_level=True)
summarize_one("BPE Byte-Level (scratch)", sample_text, bpe_tokens_byte)


[BPE Byte-Level (scratch)]
text: We love movies of all kinds. This movie was filmed in Finland by a Finnish director who ap...
num_tokens: 281
tokens[:20]: ['87', '101</w>', '108111118101</w>', '109111118105101115</w>', '111102</w>', '97108108</w>', '107', '105110', '100115</w>', '46</w>', '84104105115</w>', '109111118105101</w>', '11997115</w>', '102105108', '109', '101100</w>', '105110</w>', '70', '105110', '108']


## 8. Train a WordPiece Model (From Scratch)
We build a WordPiece vocab using imdb corpus subset first 2000 text for speed.


### wordpiece algorithm
1. word tokenize all text and count word frequency.
2. initalize vocab with [UNK] and all single characters, other than first character, all others are prefixed with ##.
3. compute frequency pair score: score(a,b) = freq(a,b)/(freq(a)*freq(b))
4. merge the pair that has the strongest score and update word splits.
5. repeat step 3 and 4 until:
   - vocab size reaches
   - no pairs remain is above or at min freq.

Tokenizations step:
1. word tokenize text
2. for each word, greedy longest match with position at zero and take the longest substring in vocab, add ## for non initial pieces.
3. If a word can't be fully segmented, use [UNK] token

In [25]:
wp_vocab = train_wordpiece(imdb_corpus[:5000], vocab_size=1000, min_freq=2)
wp_tokens_scratch = word_piece_tokenizer(sample_text, wp_vocab)
summarize_one("WordPiece (scratch)", sample_text, wp_tokens_scratch)


[WordPiece (scratch)]
text: We love movies of all kinds. This movie was filmed in Finland by a Finnish director who ap...
num_tokens: 627
tokens[:20]: ['W', '##e', 'l', '##o', '##v', '##e', 'm', '##o', '##v', '##i', '##e', '##s', 'o', '##f', 'a', '##l', '##l', 'k', '##i', '##n']


## 9. Compare Scratch vs Pretrained
Scratch models will look noisier.
The goal is to understand the mechanics, not match production tokenizers.

In [26]:
print("Pretrained BERT WordPiece (first 30):", bert_tokens[:30])
print("Scratch WordPiece (first 30):", wp_tokens_scratch[:30])
print("Pretrained GPT-2 BPE (first 30):", gpt2_tokens[:30])
print("Scratch BPE (first 30):", bpe_tokens_scratch[:30])
print("Scratch BPE byte (first 30):", bpe_tokens_byte[:30])

Pretrained BERT WordPiece (first 30): ['we', 'love', 'movies', 'of', 'all', 'kinds', '.', 'this', 'movie', 'was', 'filmed', 'in', 'finland', 'by', 'a', 'finnish', 'director', 'who', 'appears', 'to', 'be', 'trying', 'to', 'im', '##itate', 'spy', 'movies', 'from', 'james', 'bond']
Scratch WordPiece (first 30): ['W', '##e', 'l', '##o', '##v', '##e', 'm', '##o', '##v', '##i', '##e', '##s', 'o', '##f', 'a', '##l', '##l', 'k', '##i', '##n', '##d', '##s', '.', 'T', '##h', '##i', '##s', 'm', '##o', '##v']
Pretrained GPT-2 BPE (first 30): ['We', 'Ġlove', 'Ġmovies', 'Ġof', 'Ġall', 'Ġkinds', '.', 'ĠThis', 'Ġmovie', 'Ġwas', 'Ġfilmed', 'Ġin', 'ĠFinland', 'Ġby', 'Ġa', 'ĠFinnish', 'Ġdirector', 'Ġwho', 'Ġappears', 'Ġto', 'Ġbe', 'Ġtrying', 'Ġto', 'Ġimitate', 'Ġspy', 'Ġmovies', 'Ġfrom', 'ĠJames', 'ĠBond', 'Ġto']
Scratch BPE (first 30): ['W', 'e</w>', 'love</w>', 'movies</w>', 'of</w>', 'all</w>', 'k', 'in', 'ds</w>', '.</w>', 'This</w>', 'movie</w>', 'was</w>', 'fil', 'm', 'ed</w>', 'in</w>', 'F', 'in',

In [27]:
# Use a small subset for speed
subset = imdb_corpus[:2000]

metrics = {
    "bert_wordpiece": corpus_token_lengths(subset, lambda t: bert_tokenizer.tokenize(t, truncation=True, max_length=512)),
    "Scratch WordPiece": corpus_token_lengths(subset, lambda t: word_piece_tokenizer(t, wp_vocab)),
    "gpt2_byte_bpe": corpus_token_lengths(subset, lambda t: gpt2_tokenizer.tokenize(t, truncation=True, max_length=1024)),
    "Scratch BPE": corpus_token_lengths(subset, lambda t: bpe_tokenizer(t, bpe_merges)),
    "Scratch BPE byte": corpus_token_lengths(subset, lambda t: bpe_tokenizer(t, bpe_merges_byte, byte_level=True)),
}

print("\n--- Corpus Length Stats (first 2,000 IMDB texts) ---")
for name, m in metrics.items():
    print(f"{name:16} avg={m['avg_len']:<8} median={m['median_len']:<8} min={m['min_len']:<6} max={m['max_len']}")


--- Corpus Length Stats (first 2,000 IMDB texts) ---
bert_wordpiece   avg=303.84   median=230.0    min=19     max=1681
Scratch WordPiece avg=1049.2   median=791.5    min=55     max=6120
gpt2_byte_bpe    avg=290.47   median=221.0    min=16     max=1659
Scratch BPE      avg=469.73   median=349.5    min=24     max=2829
Scratch BPE byte avg=469.9    median=349.5    min=24     max=2829


# Day 3: Tie It Together (Tokenization -> LLMs -> Embeddings)

We now connect tokenization choices to what LLMs actually consume and how embeddings are formed.


## 10. The End-to-End Pipeline
A modern LLM pipeline looks like this:

1. Raw text
2. Tokenizer -> token strings
3. Tokenizer -> token IDs (integers)
4. Input formatting (special tokens, padding, masks)
5. Embedding lookup (token IDs -> vectors)
6. Positional information added
7. Transformer layers


## 11. Why Tokenization Matters for LLMs
- **Context length** is measured in tokens, not characters. Tokenization directly affects how much text fits.
- **Vocabulary size** controls embedding table size (memory and compute).
- **OOV risk** impacts robustness on messy or multilingual inputs.
- **Sequence length** affects attention cost (quadratic in length).


## 12. Embeddings: The First Learned Representation
Each token ID is used to index a row of an embedding matrix.
This is why vocab size matters: it sets the number of rows in that matrix.


In [28]:
# Tiny embedding demo: IDs -> vectors
import torch

toy_vocab_size = 10
embedding_dim = 4
emb = torch.nn.Embedding(toy_vocab_size, embedding_dim)

toy_ids = torch.tensor([1, 3, 7, 3])
toy_vectors = emb(toy_ids)
print("Token IDs:", toy_ids.tolist())
print("Embedding shape:", toy_vectors.shape)
print(toy_vectors)


Token IDs: [1, 3, 7, 3]
Embedding shape: torch.Size([4, 4])
tensor([[-0.1514, -0.9623,  0.9467, -0.6891],
        [ 2.2319, -1.8123,  0.0073, -0.1211],
        [ 0.6433, -0.9242, -1.0240, -0.2029],
        [ 2.2319, -1.8123,  0.0073, -0.1211]], grad_fn=<EmbeddingBackward0>)


## 13. Which Tokenizers Do LLMs Use?
- **BPE-family** (often byte-level) dominates modern decoder LLMs.
- **WordPiece** is common in BERT-style encoders.

Key idea: byte-level tokenizers are robust to any input, which is why they are a strong default for LLMs.


## 13A. Token Budget Example (Same Text, Different Tokenizers)
We compare how many tokens the same paragraph consumes under different tokenizers.


Note: scratch tokenizers are trained on a tiny subset, so counts can be noisier than pretrained models.


In [29]:
budget_text = (
    "This is a short paragraph to compare token budgets across tokenizers. "
    "It includes contractions, punctuation, and a URL: https://example.com."
)

bert_budget = bert_tokenizer.tokenize(budget_text, truncation=True, max_length=512)
gpt2_budget = gpt2_tokenizer.tokenize(budget_text, truncation=True, max_length=1024)
scratch_bpe_budget = bpe_tokenizer(budget_text, bpe_merges)
scratch_bpe_byte_budget = bpe_tokenizer(budget_text, bpe_merges_byte, byte_level=True)
scratch_wp_budget = word_piece_tokenizer(budget_text, wp_vocab)


print('Text:', budget_text)
print('BERT WordPiece tokens:', len(bert_budget))
print('GPT-2 Byte-BPE tokens:', len(gpt2_budget))
print('Scratch BPE tokens:', len(scratch_bpe_budget))
print('Scratch BPE byte level tokens:', len(scratch_bpe_byte_budget))
print('Scratch WordPiece tokens:', len(scratch_wp_budget))


Text: This is a short paragraph to compare token budgets across tokenizers. It includes contractions, punctuation, and a URL: https://example.com.
BERT WordPiece tokens: 36
GPT-2 Byte-BPE tokens: 31
Scratch BPE tokens: 61
Scratch BPE byte level tokens: 61
Scratch WordPiece tokens: 122


## 14. Final Summary (Tokenization -> Embeddings)
- Tokenization defines **what the model sees**.
- Input formatting defines **how the model sees it**.
- Embeddings turn token IDs into dense vectors the model can learn from.

This completes the tokenization foundation and prepares us for embeddings.
